# N067 · 网格路径回溯与访问恢复

**目标：** 区分同一路径访问和全局访问限制。

**先修：** N018, N065。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 四邻域；边界；visited；路径状态恢复；剪枝；分支因子。

## 从问题到算法

网格回溯的visited只属于当前路径，不是全局永久访问。失败分支离开格子时必须恢复，使其他路线还能使用它。单词搜索只检查指定长度；Unique Paths III则要求从起点到终点访问每个可走格恰好一次，提前碰到终点不能继续绕行。

## 最小实现

**本章接口：** `exist(board, word)`、`unique_paths_iii(grid)`

In [1]:
def exist(board,word):
    if not word: return True
    if not board or not board[0]: return False
    rows,cols=len(board),len(board[0])
    def visit(r,c,i):
        if not (0<=r<rows and 0<=c<cols) or board[r][c]!=word[i]: return False
        if i==len(word)-1: return True
        saved=board[r][c]; board[r][c]=None
        try:
            return any(visit(r+dr,c+dc,i+1) for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)])
        finally:
            board[r][c]=saved
    return any(visit(r,c,0) for r in range(rows) for c in range(cols))

def unique_paths_iii(grid):
    if not grid or not grid[0]: return 0
    rows,cols=len(grid),len(grid[0]); start=None; count=0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c]!=-1: count+=1
            if grid[r][c]==1: start=(r,c)
    if start is None: raise ValueError('a start cell is required')
    def visit(r,c,used):
        if not (0<=r<rows and 0<=c<cols) or grid[r][c]==-1: return 0
        if grid[r][c]==2: return int(used==count)
        saved=grid[r][c]; grid[r][c]=-1
        try:
            return sum(visit(r+dr,c+dc,used+1) for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)])
        finally:
            grid[r][c]=saved
    return visit(*start,1)

## 正确性、前提与复杂度

每层路径标记排除已使用格子，回溯恢复让搜索树覆盖所有简单路径。终点仅在已用格数等于可走格数时计1，不把提前到达算解。

**代价：** 单词搜索粗界O(RC·4ᴸ)、O(L)栈；全格路径搜索最坏指数、O(RC)深度。try/finally用于恢复资源，不吞掉异常。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

b=[list('ABCE'),list('SFCS'),list('ADEE')]
show_table(['网格行','字符'],list(enumerate(b)))
show_table(['目标词','能否找到','查找后首行'],[(w,exist(b,w),''.join(b[0])) for w in ['ABCCED','SEE','ABCB']])

网格行,字符
0,"['A', 'B', 'C', 'E']"
1,"['S', 'F', 'C', 'S']"
2,"['A', 'D', 'E', 'E']"


目标词,能否找到,查找后首行
ABCCED,True,ABCE
SEE,True,ABCE
ABCB,False,ABCE


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from copy import deepcopy
b=[list('ABCE'),list('SFCS'),list('ADEE')]; old=deepcopy(b)
assert exist(b,'ABCCED') and exist(b,'SEE') and not exist(b,'ABCB')
assert b==old
assert not exist([['A']],'AA')
g=[[1,0,0,0],[0,0,0,0],[0,0,2,-1]]; old=deepcopy(g)
assert unique_paths_iii(g)==2 and g==old
from functools import cache
for mask in range(16):
    g=[[1,0,0],[0,0,2]]; inner=[(0,1),(0,2),(1,0),(1,1)]
    for i,(r,c) in enumerate(inner):
        if mask>>i&1: g[r][c]=-1
    vertices=[(r,c) for r in range(2) for c in range(3) if g[r][c]!=-1]
    index={v:i for i,v in enumerate(vertices)}; end=index[(1,2)]
    @cache
    def ref(v,seen):
        if v==end: return int(seen==(1<<len(vertices))-1)
        r,c=vertices[v]; total=0
        for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)]:
            nxt=(r+dr,c+dc)
            if nxt in index and not (seen>>index[nxt]&1): total+=ref(index[nxt],seen|(1<<index[nxt]))
        return total
    old=deepcopy(g); assert unique_paths_iii(g)==ref(index[(0,0)],1<<index[(0,0)]) and g==old
print('N067: 所有本章断言通过')

N067: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 禁止修改输入实现一次。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造全局visited导致漏解的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 79. Word Search（canonical）
- 980. Unique Paths III（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。